In [10]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchaudio
import soundfile as sf
import timm
from torch.utils.data import DataLoader, Dataset
import wandb

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "."))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

classes = ["Electronic", "Experimental", "Folk", "Hip-Hop", "Instrumental", "International", "Pop", "Rock"]


In [11]:


class AudioDataset(Dataset):
    def __init__(
        self,
        path_to_csv: str,
        path_to_folder: str,
        pad_size: int,
        sr: int = 44100,
    ):
        self.csv: pd.DataFrame = pd.read_csv(path_to_csv)[["track_id", "genre"]]
        self.path_to_folder = path_to_folder
        self.pad_size = pad_size

        self.sr = sr

        self.class_to_idx = {classes[i]: i for i in range(8)}

    def __getitem__(self, index: int):
        row = self.csv.iloc[index]

        file_id = str(row["track_id"])
        if not file_id.endswith(".wav"):
            file_id += ".wav"

        path = os.path.join(self.path_to_folder, file_id)

        x, sr = sf.read(path)
        x = torch.tensor(x, dtype=torch.float32)
        
        if x.ndim == 2:
            x = x.mean(dim=1)
            
        if sr != self.sr:
            x = torchaudio.functional.resample(x, sr, self.sr)

        real_len = x.shape[0]

        if real_len < self.pad_size:
            pad_len = self.pad_size - real_len
            x = torch.nn.functional.pad(x, (0, pad_len))
        else:
            x = x[:self.pad_size]
            real_len = self.pad_size

        
        y = int(row["genre"])

        return {"x": x, "y": y, "len": real_len}

    def __len__(self):
        return self.csv.shape[0]

In [12]:
BASE_DIR = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))

duration = 3

padsize =  int(1323000 / 30 * duration)

train_dataset = AudioDataset(
    os.path.join(BASE_DIR, "data_csv", "train_meta.csv"),
    os.path.join(BASE_DIR, "data", "audio_train"), pad_size=padsize
)
val_dataset = AudioDataset(
    os.path.join(BASE_DIR, "data_csv", "val_meta.csv"),
    os.path.join(BASE_DIR, "data", "audio_val"), pad_size=padsize
)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

In [13]:
def compute_log_melspectrogram(wav_batch, lens, sr, device="cpu"):
    n_fft = 1024
    win_length = 1024
    hop_length = 256
    n_mels = 64

    wav_batch = wav_batch.to(device)

    windows = wav_batch.unfold(dimension=1, size=win_length, step=hop_length)
    filter = torch.hann_window(win_length, device=device)
    windows_with_applied_filter = windows * filter[None, None, :]

    fft_features = torch.fft.fft(windows_with_applied_filter, n=n_fft, dim=-1)[:, :, : n_fft // 2 + 1]
    fft_magnitudes = torch.abs(fft_features ** 2)

    melscale = torchaudio.transforms.MelScale(n_mels=n_mels, sample_rate=sr, n_stft=n_fft // 2 + 1).to(device)
    mel_spectrogram = melscale(fft_magnitudes.transpose(1, 2))
    logmel_spectrogram = torch.log(mel_spectrogram.clamp(1e-5)) 

    logmel_spectrogram = logmel_spectrogram.unsqueeze(1)
    
    logmel_spectrogram = F.interpolate(
        logmel_spectrogram, 
        size=(224, 224), 
        mode='bilinear', 
        align_corners=False
    )
    
    logmel_spectrogram = logmel_spectrogram.repeat(1, 3, 1, 1)

    return logmel_spectrogram, lens // hop_length



In [14]:
model = timm.create_model(
    'tf_efficientnetv2_s',
    pretrained=True,
    num_classes=8 
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = torch.nn.CrossEntropyLoss()


In [15]:
def train_and_eval(model, train_loader, val_loader, epochs=5):
    wandb.init(project="gp5-audio-classification", name="tf_efficientnetv2_s")
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            x, lens = compute_log_melspectrogram(batch["x"], batch["len"], sr=44100, device=device)
            y = batch["y"].to(device)
            
            optimizer.zero_grad()
            probs = model(x)
            loss = criterion(probs, y)
            loss.backward()
            optimizer.step()
            
            acc = (probs.argmax(dim=-1) == y).float().mean().item()
            wandb.log({"train/loss": loss.item(), "train/accuracy": acc})
            
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for batch in val_loader:
                x, lens = compute_log_melspectrogram(batch["x"], batch["len"], sr=44100, device=device)
                y = batch["y"].numpy()
                probs = model(x)
                val_preds.extend(probs.argmax(dim=-1).cpu().numpy())
                val_targets.extend(y)
        
        val_acc = np.mean(np.array(val_preds) == np.array(val_targets))
        print(f"Epoch {epoch}: Val Accuracy = {val_acc:.4f}")
        wandb.log({"epoch": epoch, "val/accuracy": val_acc})

train_and_eval(model, train_dataloader, val_dataloader)


wandb: Currently logged in as: wh1t33 (wh1t33-hse) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train/accuracy,▁▁▂▄▄▇▆▄█▇▅▆▆▅▂▃▂▄▄█▆▆▄▅▅▂▅▅▇▇▆▅█▅▅▆▆▆▇▅
train/loss,█▇▇▅▇▅▅▄▂▂▂▂▂▂▂▃▂▂▁▂▁▂▂▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▂▂
train/accuracy,0.375
train/loss,1.79403


Epoch 0: Val Accuracy = 0.5544
Epoch 1: Val Accuracy = 0.5256
Epoch 2: Val Accuracy = 0.4900
Epoch 3: Val Accuracy = 0.5000
Epoch 4: Val Accuracy = 0.5300
